In [ ]:
# ==============================================================================
# BƯỚC 1: CÀI ĐẶT MÔI TRƯỜNG

import requests
import time
import pandas as pd
import pandas_ta as ta
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from datetime import datetime
from hmmlearn.hmm import GaussianHMM
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# BƯỚC 2: CẤU HÌNH CHIẾN LƯỢC (CONFIG)
# ==============================================================================
CONFIG = {
    'SYMBOL': 'BTCUSDT',
    'INTERVAL': '1h',        # Khung thời gian giao dịch
    'HTF_INTERVAL': '4h',    # Khung thời gian HMM (Vĩ mô)
    'START_DATE': '01/01/2021',
    'END_DATE': '31/12/2025',
    'INITIAL_CAPITAL': 10000.0,

    # --- HMM Parameters ---
    'HMM_WINDOW': 1000,      # Cửa sổ dữ liệu để HMM học
    'RETRAIN_STEP': 24,      # Học lại sau mỗi 24 nến (1 ngày) để tăng tốc
    'HMM_STABILITY': 3,      # Cần 3 nến HMM đồng thuận mới tin (Anti-Flicker)

    # --- Ichimoku Parameters ---
    'TENKAN': 9,
    'KIJUN': 26,
    'SPAN_B': 52,
    'DISPLACE': 26,

    # --- Management ---
    'SL_ATR_MULT': 1.5,      # Cắt lỗ theo ATR
    'RISK_PER_TRADE': 0.02,  # Rủi ro 2% vốn mỗi lệnh

    # Costs
    'FEE_RATE': 0.0004,
    'SLIPPAGE': 0.0002
}

# ==============================================================================
# BƯỚC 3: MODULE TẢI DỮ LIỆU (Robust Fetcher)
# ==============================================================================
def str_to_millis(date_str: str) -> int:
    dt = datetime.strptime(date_str, "%d/%m/%Y")
    return int(dt.timestamp() * 1000)

def fetch_binance_kline_df(symbol: str, interval: str, start_str: str, end_str: str) -> pd.DataFrame:
    base_url = "https://data-api.binance.vision/"
    req_symbol = symbol.upper()

    # Kiểm tra IP Mỹ
    try:
        test = requests.get(f"{base_url}/api/v3/time", timeout=5)
        if test.status_code == 451:
            base_url = "https://api.binance.us"
            if req_symbol == "BTCUSDT": req_symbol = "BTCUSD"
    except:
        base_url = "https://api.binance.us"
        req_symbol = "BTCUSD"

    url = base_url + "/api/v3/klines"
    start_time = str_to_millis(start_str)

    if end_str == datetime.now().strftime("%d/%m/%Y"): end_time = int(time.time() * 1000)
    else: end_dt = datetime.strptime(end_str, "%d/%m/%Y").replace(hour=23, minute=59, second=59); end_time = int(end_dt.timestamp() * 1000)

    all_klines = []
    print(f"⏳ Downloading {req_symbol} ({interval})...")

    while start_time < end_time:
        params = {"symbol": req_symbol, "interval": interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
        try:
            r = requests.get(url, params=params, timeout=10)
            if r.status_code != 200: break
            data = r.json()
            if not data: break
            all_klines.extend(data)
            start_time = data[-1][0] + 1
            time.sleep(0.05)
        except: break

    if not all_klines: return pd.DataFrame()
    df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
    df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df.set_index("timestamp", inplace=True)
    df = df.astype(float)
    return df

# ==============================================================================
# BƯỚC 4: HMM ENGINE (OPTIMIZED LAZY TRAINING)
# ==============================================================================
def train_hmm_optimized(df_htf):
    """Huấn luyện HMM theo cơ chế Lazy Retraining để tối ưu tốc độ"""
    print("🧠 Training HMM Model (Fast Mode)...")

    # 1. Feature Engineering
    df_htf['Log_Ret'] = np.log(df_htf['Close'] / df_htf['Close'].shift(1) + 1e-9)
    df_htf['ATR'] = ta.atr(df_htf['High'], df_htf['Low'], df_htf['Close'], length=14)

    # Chuẩn hóa dữ liệu (Normalize) - Quan trọng để HMM hội tụ
    window = CONFIG['HMM_WINDOW']
    df_htf['ATR_Norm'] = (df_htf['ATR'] - df_htf['ATR'].rolling(window).mean()) / df_htf['ATR'].rolling(window).std()

    df_clean = df_htf.dropna().reset_index()
    X = df_clean[['Log_Ret', 'ATR_Norm']].values

    # 2. Lazy Training Loop
    states_pred = np.zeros(len(df_clean), dtype=int)
    model = GaussianHMM(n_components=3, covariance_type="full", n_iter=100, tol=0.01, random_state=42)
    last_model = None

    for t in tqdm(range(window, len(df_clean))):
        # Chỉ học lại sau mỗi RETRAIN_STEP (ví dụ 24 nến)
        if t % CONFIG['RETRAIN_STEP'] == 0 or last_model is None:
            train_data = X[t-window : t]
            try:
                # Warm Start: Dùng kiến thức cũ để học nhanh hơn
                if last_model:
                    model.startprob_ = last_model.startprob_
                    model.transmat_ = last_model.transmat_
                    model.means_ = last_model.means_
                    model.covars_ = last_model.covars_
                    model.init_params = ''

                model.fit(train_data)
                last_model = model
            except: pass

        # Predict: Luôn chạy ở mỗi nến để có trạng thái realtime
        if last_model:
            try:
                current_feat = X[t].reshape(1, -1)
                hidden_state = last_model.predict(current_feat)[0]

                # Mapping: Sort state theo Mean Return để cố định ý nghĩa (0: Sideway, 1: Bull, 2: Bear)
                means = last_model.means_[:, 0]
                sorted_idx = np.argsort(means) # [Thấp nhất, Giữa, Cao nhất]
                mapper = {sorted_idx[0]: 2, sorted_idx[1]: 0, sorted_idx[2]: 1} # 2=Bear, 0=Side, 1=Bull

                states_pred[t] = mapper[hidden_state]
            except: states_pred[t] = 0

    df_clean['HMM_State'] = states_pred
    # Trả về Series có DateTime Index để merge
    return df_clean.set_index('timestamp')[['HMM_State']]

# ==============================================================================
# BƯỚC 5: BACKTEST ENGINE (HMM + ICHIMOKU)
# ==============================================================================
def run_hybrid_strategy(df_1h, df_hmm):
    print("⚔️ Running Strategy Logic...")
    c = CONFIG

    # --- A. DATA MERGING ---
    # Merge trạng thái HMM từ khung lớn (4H) vào khung nhỏ (1H)
    df = df_1h.join(df_hmm, how='left')
    df['HMM_State'] = df['HMM_State'].fillna(method='ffill').fillna(0)

    # --- B. INDICATORS ---
    # Ichimoku
    roll_max = lambda x, n: x.rolling(n).max()
    roll_min = lambda x, n: x.rolling(n).min()

    df['Tenkan'] = (roll_max(df['High'], c['TENKAN']) + roll_min(df['Low'], c['TENKAN'])) / 2
    df['Kijun']  = (roll_max(df['High'], c['KIJUN']) + roll_min(df['Low'], c['KIJUN'])) / 2

    # Mây hiện tại (đã shift)
    df['SpanA'] = ((df['Tenkan'] + df['Kijun']) / 2).shift(c['DISPLACE'])
    df['SpanB'] = ((roll_max(df['High'], c['SPAN_B']) + roll_min(df['Low'], c['SPAN_B'])) / 2).shift(c['DISPLACE'])

    df['ATR'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)
    df.dropna(inplace=True)

    # --- C. EXECUTION ---
    capital = c['INITIAL_CAPITAL']
    position = 0; entry_price = 0; sl_price = 0
    equity = [capital] * len(df)
    trades = []

    # Numpy Arrays
    O=df['Open'].values; H=df['High'].values; L=df['Low'].values; C=df['Close'].values
    Kijun=df['Kijun'].values; SpanA=df['SpanA'].values; SpanB=df['SpanB'].values
    HMM=df['HMM_State'].values; ATR=df['ATR'].values
    Times = df.index

    for i in range(3, len(df)):
        current_state = HMM[i]

        # 1. Exit Logic
        if position != 0:
            exit_signal = False; reason = ""; exit_price = C[i]

            # Anti-Flicker: HMM phải đổi trạng thái bền vững (3 nến liên tiếp) mới kích hoạt Emergency Exit
            hmm_stable_change = (HMM[i] == HMM[i-1] == HMM[i-2]) and (HMM[i] != (1 if position == 1 else 2))

            if hmm_stable_change:
                exit_signal = True; reason = "HMM Change"

            # Kijun Trailing Exit
            elif (position == 1 and C[i] < Kijun[i]) or (position == -1 and C[i] > Kijun[i]):
                exit_signal = True; reason = "Kijun Cross"

            # SL Hit
            elif (position == 1 and L[i] <= sl_price) or (position == -1 and H[i] >= sl_price):
                exit_signal = True; reason = "SL Hit"; exit_price = sl_price

            if exit_signal:
                # Calc PnL
                if position == 1:
                    real_exit = min(O[i], exit_price) if "SL" in reason else exit_price
                    real_exit *= (1 - c['SLIPPAGE'])
                    pnl = (real_exit - entry_price) / entry_price
                else:
                    real_exit = max(O[i], exit_price) if "SL" in reason else exit_price
                    real_exit *= (1 + c['SLIPPAGE'])
                    pnl = (entry_price - real_exit) / entry_price

                net_pnl = pnl - (c['FEE_RATE'] * 2)
                capital *= (1 + net_pnl)
                trades.append({'PnL': net_pnl, 'Type': position, 'Reason': reason})
                position = 0
            else:
                # Trailing Stop
                if position == 1: sl_price = max(sl_price, Kijun[i])
                else: sl_price = min(sl_price, Kijun[i])

        # 2. Entry Logic
        if position == 0:
            # HMM Filter: Chỉ vào lệnh khi HMM ổn định (3 nến cùng màu)
            hmm_is_bull = (HMM[i] == 1 and HMM[i-1] == 1 and HMM[i-2] == 1)
            hmm_is_bear = (HMM[i] == 2 and HMM[i-1] == 2 and HMM[i-2] == 2)

            cloud_top = max(SpanA[i], SpanB[i])
            cloud_bot = min(SpanA[i], SpanB[i])

            # Long Setup
            if hmm_is_bull:
                if C[i] > cloud_top: # Giá trên mây
                    # Pullback: Low chạm Kijun nhưng Close giữ trên Kijun
                    if L[i] <= Kijun[i] and C[i] > Kijun[i]:
                        position = 1
                        entry_price = C[i] * (1 + c['SLIPPAGE'])
                        sl_price = cloud_bot - (c['SL_ATR_MULT'] * ATR[i])

            # Short Setup
            elif hmm_is_bear:
                if C[i] < cloud_bot: # Giá dưới mây
                    if H[i] >= Kijun[i] and C[i] < Kijun[i]:
                        position = -1
                        entry_price = C[i] * (1 - c['SLIPPAGE'])
                        sl_price = cloud_top + (c['SL_ATR_MULT'] * ATR[i])

        equity[i] = capital

    return pd.Series(equity, index=df.index), pd.DataFrame(trades)

# ==============================================================================
# BƯỚC 6: REPORTING (SHARPE RATIO ADDED)
# ==============================================================================
def calculate_metrics(equity, trades_df):
    final = equity.iloc[-1]
    profit = (final - CONFIG['INITIAL_CAPITAL']) / CONFIG['INITIAL_CAPITAL']
    dd = (equity - equity.cummax()) / equity.cummax()

    # Sharpe Ratio (Annualized)
    returns = equity.pct_change().dropna()
    # 1 năm có 365*24 nến 1H
    sharpe = (returns.mean() / returns.std()) * np.sqrt(365 * 24) if returns.std() != 0 else 0

    winrate = len(trades_df[trades_df['PnL'] > 0]) / len(trades_df) if not trades_df.empty else 0

    print(f"\n📊 KẾT QUẢ BACKTEST: {CONFIG['SYMBOL']}")
    print(f"{'='*30}")
    print(f"{'Final Equity:':>15} ${final:,.2f}")
    print(f"{'Total Return:':>15} {profit:.2%}")
    print(f"{'Max Drawdown:':>15} {dd.min():.2%}")
    print(f"{'Sharpe Ratio:':>15} {sharpe:.2f}")
    print(f"{'Total Trades:':>15} {len(trades_df)}")
    print(f"{'Winrate:':>15} {winrate:.2%}")
    print(f"{'='*30}")

    return equity, dd

# ==============================================================================
# MAIN RUN
# ==============================================================================
if __name__ == "__main__":
    # 1. Fetch Data
    df_1h = fetch_binance_kline_df(CONFIG['SYMBOL'], CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])

    if not df_1h.empty:
        # 2. Prepare HTF Data for HMM
        df_4h = df_1h.resample(CONFIG['HTF_INTERVAL']).agg({
            'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'
        }).dropna()

        # 3. Train HMM
        df_hmm_states = train_hmm_optimized(df_4h)

        # 4. Run Backtest
        equity, trades = run_hybrid_strategy(df_1h, df_hmm_states)

        # 5. Show Results
        if not trades.empty:
            calculate_metrics(equity, trades)

            # Plotting
            plt.figure(figsize=(15, 10))
            plt.subplot(2, 1, 1)
            plt.plot(equity, label='Equity', color='#00BCD4')
            plt.title('HMM + Ichimoku Equity Curve')
            plt.legend(); plt.grid(True, alpha=0.3)

            plt.subplot(2, 1, 2)
            # Visualize HMM States on recent data (last 500 bars)
            recent_hmm = df_hmm_states.iloc[-500:]
            recent_price = df_4h['Close'].iloc[-500:]

            plt.plot(recent_price, color='black', alpha=0.6)
            ax = plt.gca()
            trans = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)

            for i in range(len(recent_hmm) - 1):
                state = recent_hmm['HMM_State'].iloc[i]
                color = 'white'
                if state == 1: color = 'lightgreen' # Bull
                elif state == 2: color = 'lightcoral' # Bear
                plt.fill_between(recent_hmm.index[i:i+2], 0, 1, facecolor=color, alpha=0.5, transform=trans)

            plt.title('HMM States (Green=Bull, Red=Bear, White=Sideway)')
            plt.show()
        else:
            print("⚠️ Không có lệnh nào được khớp!")

Found existing installation: numpy 2.2.6
Uninstalling numpy-2.2.6:
  Successfully uninstalled numpy-2.2.6
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.8 MB/s eta 0:00:00
ERROR: Cannot install numpy<2.0.0, pandas-ta==0.4.67b0 and pandas-ta==0.4.71b0 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
